# LLM & Neural Model Optimization for Production — Python Lab

**Goal:** understand and implement the major ways neural networks and LLMs are made smaller, faster, cheaper, and easier to serve in production.

This notebook is intentionally **broader than edge AI**. Edge/mobile deployment remains an important target, but the same optimization ideas also matter for:

- GPU inference servers
- Kubernetes / vLLM-style serving
- cloud inference endpoints
- CPU inference
- multimodal systems
- latency-sensitive APIs
- memory-constrained deployments
- edge / mobile / NPU environments

The notebook follows one rule:

> **For every important optimization concept, understand what changes under the hood, then run Python that demonstrates the change.**

Major topics include tensor/matrix foundations, pruning, distillation, low-rank approximation, quantization, NAS, LLM-specific optimization, KV cache, compilation, partitioning, edge runtimes, and production benchmarking.

**Core distinction**

- **Pruning** changes which computation exists.
- **Quantization** changes numerical precision.
- **Distillation** transfers behavior into a different/smaller model.
- **Low-rank factorization** approximates large matrices with smaller matrices.
- **NAS** reallocates architectural capacity.
- **Compilation/kernel optimization** changes how the computation executes on hardware.


## 0. The complete production optimization pipeline

```text
                     ACCURACY-CAPABLE BASELINE
                               │
                     profile actual bottleneck
                               │
          ┌────────────────────┼────────────────────┐
          ▼                    ▼                    ▼
      COMPUTE COST        MEMORY / BANDWIDTH      LATENCY
          │                    │                    │
          └────────────────────┼────────────────────┘
                               ▼
                    architecture / NAS
                               │
                    structured pruning
             channels / neurons / heads / blocks
                               │
                       distillation
                               │
                    low-rank factorization
                               │
                 quantization / mixed precision
                               │
              graph optimization / kernel fusion
                               │
                   runtime / serving engine
                               │
      ┌──────────────┬─────────┼──────────────┬──────────────┐
      ▼              ▼         ▼              ▼              ▼
     CPU            GPU       NPU         cloud/K8s         edge
                vLLM/Triton                 serving     ExecuTorch/TVM
```

There is **no universal ordering**. The right sequence is measured against the actual model, workload, and hardware.


In [ ]:
# 1. Environment
# Recommended:
#   pip install torch torchvision torch-pruning torchprofile matplotlib pandas
#
# Optional edge/compiler environments:
#   pip install executorch
#   pip install apache-tvm
#
# Exact edge backend packages vary by target hardware.

import copy
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

print("PyTorch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 1. Matrix and tensor foundations — why GPUs matter

A GPU is **not physically designed around a 3×3 CNN kernel**. A 3×3 kernel is simply one workload that maps well onto GPU arithmetic.

```text
scalar       7

vector       [1, 2, 3]

matrix       [[1, 2],
              [3, 4]]

3-D tensor   [channels, height, width]

4-D tensor   [batch, channels, height, width]

LLM tensor   [batch, sequence, hidden_dimension]
```

The central operation is usually some version of **multiply-accumulate**:

```text
D = A @ B + C
```

Transformers make this especially visible:

```text
Q = X @ Wq
K = X @ Wk
V = X @ Wv

scores = Q @ Kᵀ
output = softmax(scores) @ V
```

The advantage is not simply adding another dimension. The advantage is turning the workload into **large regular blocks of arithmetic that can execute massively in parallel**.


In [ ]:
# Matrix multiplication: a core primitive behind neural-network compute.

import torch
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

A = torch.tensor([[1., 2.],
                  [3., 4.]], device=device)
B = torch.tensor([[5., 6.],
                  [7., 8.]], device=device)

print("A @ B =\n", A @ B)

N = 1024
A_big = torch.randn(N, N, device=device)
B_big = torch.randn(N, N, device=device)

if device == "cuda":
    torch.cuda.synchronize()

t0 = time.perf_counter()
C_big = A_big @ B_big

if device == "cuda":
    torch.cuda.synchronize()

print("result shape:", C_big.shape)
print(f"{N}x{N} matrix multiply: {time.perf_counter() - t0:.4f}s")


### From a matrix to a convolution channel

For:

```python
nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
```

the weight tensor has shape:

```text
[Cout, Cin, Kh, Kw]
[  32,   3,  3,  3]
```

There are **32 output filters/channels**.

Each filter spans the 3 input channels and a 3×3 spatial region:

```text
3 × 3 × 3 = 27 learned weights per output filter
```

Each output filter produces one output feature map.


In [ ]:
import torch
import torch.nn as nn

conv = nn.Conv2d(3, 32, kernel_size=3, padding=1)
x = torch.randn(1, 3, 224, 224)
y = conv(x)

print("input: ", x.shape)
print("weight:", conv.weight.shape)
print("output:", y.shape)

filter_17 = conv.weight[17]
print("filter 17 shape:", filter_17.shape)
print("weights in filter 17:", filter_17.numel())


### Kernel size, padding, and stride

- `kernel_size=3` = a **3×3 spatial window**, not three GPU threads.
- `padding=1` = a one-element spatial border around the input.
- `stride=1` = move one position at a time.

With a 3×3 kernel, padding 1 and stride 1 commonly preserve height and width.


In [ ]:
conv_same = nn.Conv2d(3, 32, kernel_size=3, padding=1, stride=1)
conv_no_pad = nn.Conv2d(3, 32, kernel_size=3, padding=0, stride=1)

x = torch.randn(1, 3, 224, 224)

print("input:       ", x.shape)
print("padding=1:   ", conv_same(x).shape)
print("padding=0:   ", conv_no_pad(x).shape)


## 2. A tiny CNN we can actually manipulate

Using a small CNN makes the structural effects visible. The same ideas transfer to ResNet, MobileNet, ViT and transformer blocks.

### What a convolution channel means

For:

```text
Conv2d(in_channels=3, out_channels=32)
```

there are **32 output filters/channels**.

If we remove output channel 17, we have removed:

```text
filter 17
      ↓
feature map 17
      ↓
the corresponding input channel in the next layer
```

That is **structured channel pruning**.

By contrast, setting a single weight such as:

```text
W[17, 2, 3, 3] = 0
```

is **unstructured pruning**. The tensor shape does not shrink, so generic hardware may see little or no speedup.


In [ ]:
class EdgeCNN(nn.Module):
    def __init__(self, c1=32, c2=64, c3=128, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, c1, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(c1)
        self.conv2 = nn.Conv2d(c1, c2, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(c2)
        self.conv3 = nn.Conv2d(c2, c3, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(c3)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(c3, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x).flatten(1)
        return self.fc(x)

model = EdgeCNN().to(device)
model


In [ ]:
# 3. Baseline parameter and approximate storage measurements

def count_parameters(m):
    return sum(p.numel() for p in m.parameters())

def parameter_bytes(m, bits=32):
    return count_parameters(m) * bits // 8

def summarize_model(m, name="model"):
    params = count_parameters(m)
    return {
        "model": name,
        "parameters": params,
        "params_M": params / 1e6,
        "FP32_MB": parameter_bytes(m) / (1024**2),
    }

baseline = summarize_model(model, "baseline")
pd.DataFrame([baseline])


## 3.1 Compression math

For a dense model with `N` parameters:

- FP32 storage ≈ `4N` bytes
- INT8 storage ≈ `N` bytes
- INT4 storage ≈ `0.5N` bytes

Approximate ideal weight-only compression ratios:

```text
FP32 → INT8 : 4×
FP32 → INT4 : 8×
```

Real files are larger because of scales, zero-points, metadata, alignment and sometimes duplicated tensors.

**Pruning is different:** if 30% of weights are zero but the model is still stored as a dense FP32 tensor, storage may remain approximately unchanged. Sparse encoding or physically removing structured dimensions is required to realize the storage/compute benefit.


# 4. Unstructured pruning — individual weights disappear/deactivate

This is the first place to see exactly what pruning means.

`torch.nn.utils.prune` can apply a mask such as:

```text
W = W_original × mask
```

where:

```text
mask[i] = 0 → weight is disabled
mask[i] = 1 → weight remains active
```

The important catch:

> A zero weight is **not automatically a smaller neural network**.

It is sparse computation only if the storage format and kernel/backend exploit that sparsity.


In [ ]:
import torch.nn.utils.prune as prune

pruned = copy.deepcopy(model)

layer = pruned.conv2
before_nonzero = torch.count_nonzero(layer.weight).item()
total = layer.weight.numel()

# Remove 40% of the lowest-magnitude individual weights.
prune.l1_unstructured(layer, name="weight", amount=0.40)

after_nonzero = torch.count_nonzero(layer.weight).item()

print("Total weights:", total)
print("Nonzero weights:", after_nonzero)
print("Observed sparsity:", 1 - after_nonzero / total)

# Make the mask permanent.
prune.remove(layer, "weight")
print("Physical tensor shape:", tuple(layer.weight.shape))


### Visualizing individual weight removal

The left/right comparison below shows the core difference between dense and sparse tensors.

For real edge acceleration, ask:

1. Does the target accelerator support sparse kernels?
2. What sparsity pattern is supported — arbitrary, block, 2:4, etc.?
3. Does the compiler recognize the pattern?
4. Does memory traffic actually fall?
5. Does latency improve on the target device?

Those questions are more important than the percentage of zeros by itself.


In [ ]:
# Visualize one output filter before/after unstructured pruning.
original = model.conv2.weight.detach().cpu()
sparse = pruned.conv2.weight.detach().cpu()

f = 0
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(original[f, 0].numpy(), cmap="gray")
plt.title("Original filter")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(sparse[f, 0].numpy(), cmap="gray")
plt.title("After unstructured pruning")
plt.axis("off")
plt.show()


# 5. Structured pruning — physically removing computation

Structured pruning is more important than simply setting scattered weights to zero because it can **change tensor dimensions and remove entire units of computation**.

For a CNN:

```text
Conv1 output filter 17
        ↓
feature map/channel 17
        ↓
Conv2 input channel 17
```

If output channel 17 is physically removed from `Conv1`, the matching input channel must also be removed from `Conv2`.

That changes:

```text
Conv1: 3  → 32       Conv2: 32 → 64
```

into:

```text
Conv1: 3  → 31       Conv2: 31 → 64
```

This is a **graph/tensor transformation**, not merely zeroing values.

For LLMs the same structural idea applies to different objects:

```text
CNN          → filters / feature-map channels
Transformer  → attention heads / MLP neurons / layers
MoE          → experts
```


In [ ]:
# Physical structured pruning example:
# remove Conv1 output channel 17 AND the matching Conv2 input channel.

import torch
import torch.nn as nn

conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

remove_channel = 17
keep = [i for i in range(32) if i != remove_channel]

new_conv1 = nn.Conv2d(3, 31, kernel_size=3, padding=1)
new_conv2 = nn.Conv2d(31, 64, kernel_size=3, padding=1)

with torch.no_grad():
    # Conv1: remove output filter/channel 17.
    new_conv1.weight.copy_(conv1.weight[keep])
    new_conv1.bias.copy_(conv1.bias[keep])

    # Conv2: remove the corresponding INPUT channel 17.
    new_conv2.weight.copy_(conv2.weight[:, keep])
    new_conv2.bias.copy_(conv2.bias)

print("OLD Conv1:", conv1.weight.shape)      # [32, 3, 3, 3]
print("NEW Conv1:", new_conv1.weight.shape)  # [31, 3, 3, 3]

print("OLD Conv2:", conv2.weight.shape)      # [64, 32, 3, 3]
print("NEW Conv2:", new_conv2.weight.shape)  # [64, 31, 3, 3]

x = torch.randn(1, 3, 64, 64)
y = new_conv2(new_conv1(x))
print("network still runs:", y.shape)


### Why this actually reduces work

For a convolution:

```text
MACs ≈ H × W × Cin × Cout × K²
```

So reducing `Cin` and `Cout` reduces arithmetic directly.

This is the important production distinction:

```text
zero a weight          → value disappears
remove a channel       → tensor dimension and downstream computation disappear
```


In [ ]:
def channel_l1_scores(conv):
    # One score per output channel/filter.
    w = conv.weight.detach()
    return w.abs().sum(dim=(1, 2, 3))

def channels_to_keep(conv, keep_ratio=0.75):
    scores = channel_l1_scores(conv)
    k = max(1, int(len(scores) * keep_ratio))
    return torch.topk(scores, k=k).indices.sort().values

scores = channel_l1_scores(model.conv2)
keep = channels_to_keep(model.conv2, 0.75)

print("Original output channels:", model.conv2.out_channels)
print("Keeping:", len(keep))
print("Pruning:", model.conv2.out_channels - len(keep))
print("Lowest-scoring channels:", torch.argsort(scores)[:10].tolist())


In [ ]:
# A compact, runnable channel-pruning transformation for our sequential CNN.
# This physically constructs a smaller network instead of leaving zero channels behind.

def prune_edgecnn_channels(src, keep1=0.75, keep2=0.75, keep3=0.75):
    # Select channels using L1 filter importance.
    idx1 = channels_to_keep(src.conv1, keep1)
    idx2 = channels_to_keep(src.conv2, keep2)
    idx3 = channels_to_keep(src.conv3, keep3)

    new = EdgeCNN(
        c1=len(idx1), c2=len(idx2), c3=len(idx3),
        num_classes=src.fc.out_features
    ).to(next(src.parameters()).device)

    with torch.no_grad():
        # Conv1: select output filters.
        new.conv1.weight.copy_(src.conv1.weight[idx1])
        new.conv1.bias.copy_(src.conv1.bias[idx1])
        new.bn1.weight.copy_(src.bn1.weight[idx1])
        new.bn1.bias.copy_(src.bn1.bias[idx1])
        new.bn1.running_mean.copy_(src.bn1.running_mean[idx1])
        new.bn1.running_var.copy_(src.bn1.running_var[idx1])

        # Conv2: select output AND the surviving input channels.
        new.conv2.weight.copy_(src.conv2.weight[idx2][:, idx1])
        new.conv2.bias.copy_(src.conv2.bias[idx2])
        new.bn2.weight.copy_(src.bn2.weight[idx2])
        new.bn2.bias.copy_(src.bn2.bias[idx2])
        new.bn2.running_mean.copy_(src.bn2.running_mean[idx2])
        new.bn2.running_var.copy_(src.bn2.running_var[idx2])

        # Conv3.
        new.conv3.weight.copy_(src.conv3.weight[idx3][:, idx2])
        new.conv3.bias.copy_(src.conv3.bias[idx3])
        new.bn3.weight.copy_(src.bn3.weight[idx3])
        new.bn3.bias.copy_(src.bn3.bias[idx3])
        new.bn3.running_mean.copy_(src.bn3.running_mean[idx3])
        new.bn3.running_var.copy_(src.bn3.running_var[idx3])

        # FC input follows conv3 output channels.
        new.fc.weight.copy_(src.fc.weight[:, idx3])
        new.fc.bias.copy_(src.fc.bias)

    return new

structured = prune_edgecnn_channels(model, 0.75, 0.75, 0.75)

comparison = pd.DataFrame([
    summarize_model(model, "dense baseline"),
    summarize_model(structured, "structured channels -25%"),
])
comparison


### Why structured pruning can reduce computation

For a convolution:

```text
MACs ≈ H × W × Cin × Cout × K²
```

So reducing both `Cin` and `Cout` has a multiplicative effect.

If both dimensions are reduced to 75%:

```text
0.75 × 0.75 = 0.5625
```

or roughly **56% of the original convolution MACs** before accounting for changes elsewhere.

This is the key difference from simply zeroing 25% of weights.


# 6. Knowledge distillation — recover accuracy after compression

Distillation does not directly delete weights.

Instead:

```text
large teacher
     │
     ├── logits / soft targets
     ▼
small student
     │
     └── learns task labels + teacher behavior
```

A common objective is:

```text
Loss = α × CE(student, labels)
     + β × KL(student/T, teacher/T) × T²
```

where `T` is the temperature.

This is especially useful after NAS or structured pruning because the smaller student can regain some of the accuracy lost by compression.


In [ ]:
# Distillation implementation.
# This uses synthetic data so the notebook is self-contained.
# Replace this DataLoader with CIFAR-10/ImageNet/custom data in a real project.

teacher = copy.deepcopy(model).eval()
student = copy.deepcopy(structured).train()

def distillation_loss(student_logits, teacher_logits, labels, T=4.0, alpha=0.5):
    hard = F.cross_entropy(student_logits, labels)
    soft = F.kl_div(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1),
        reduction="batchmean",
    ) * (T * T)
    return alpha * soft + (1 - alpha) * hard

x = torch.randn(16, 3, 32, 32, device=device)
y = torch.randint(0, 10, (16,), device=device)

with torch.no_grad():
    teacher_logits = teacher(x)

student_logits = student(x)
loss = distillation_loss(student_logits, teacher_logits, y)
print("Example distillation loss:", float(loss))


# 7. Quantization — lower numerical precision

Quantization changes representation:

```text
FP32 weight
   ↓
scale + zero-point
   ↓
INT8 representation
```

Two important workflows:

### PTQ — Post Training Quantization
Train FP32 → calibrate → quantize.

Fast and simple, but accuracy can drop.

### QAT — Quantization Aware Training
Train while simulating quantization.

Usually more work, but lets the model adapt to quantization noise.

For edge systems, **hybrid quantization** is often useful: sensitive layers remain higher precision while less-sensitive layers use INT8/INT4.

ExecuTorch currently documents both PTQ/QAT concepts and backend-specific quantization paths; its runtime can lower exported graphs to CPU/GPU/NPU/DSP-style backends. citeturn0search0turn0search2


In [ ]:
# 8. Simple, runnable dynamic quantization example.
# Dynamic quantization is particularly straightforward for Linear layers.
# For Conv-heavy edge models, use a backend-specific static/PT2E/QAT flow.

quantized_student = torch.ao.quantization.quantize_dynamic(
    copy.deepcopy(student).cpu(),
    {nn.Linear},
    dtype=torch.qint8
)

print("FP32 student:", summarize_model(student.cpu(), "student FP32"))
print("Dynamic INT8 Linear:", summarize_model(quantized_student, "student dynamic INT8"))


### Quantization choices

| Technique | Typical idea | Main benefit | Main issue |
|---|---|---|---|
| FP16/BF16 | 16-bit floating point | easy memory/compute reduction | hardware dependent |
| INT8 PTQ | 8-bit weights/activations | mature edge option | calibration sensitivity |
| INT8 QAT | train with quantization effects | better accuracy retention | training cost |
| INT4 weight-only | 4-bit weights | major model-size reduction | kernels/backend and accuracy sensitivity |
| mixed precision | different precision by layer | accuracy/latency tradeoff | more deployment complexity |

For LLMs, weight-only INT4 and mixed-precision approaches become especially important because weights dominate memory. ExecuTorch's current LLM documentation also exposes low-bit TorchAO configurations such as INT4/INT8 and mixed activation/weight modes. citeturn0search1turn0search10


# 9. Neural Architecture Search — redistribute capacity

Pruning asks:

> **Which existing capacity can I remove?**

NAS asks:

> **Where should the remaining capacity go?**

A search space can vary:

```text
channels:   16 / 24 / 32 / 48 / 64
depth:       1 / 2 / 3 / 4
kernel:      3 / 5 / 7
expansion:   2 / 4 / 6
attention:   enabled / disabled
precision:   FP16 / INT8
```

For edge NAS, the objective is often multi-objective:

```text
maximize accuracy
minimize latency
minimize memory
minimize energy
minimize flash size
```

This can be represented as a Pareto problem rather than a single "best" model.


In [ ]:
# 10. Tiny brute-force NAS example.
# This is deliberately simple: a real project may use Optuna, NNI,
# evolutionary search, differentiable NAS, or hardware-in-the-loop search.

from itertools import product

search_space = {
    "c1": [16, 24, 32],
    "c2": [32, 48, 64],
    "c3": [64, 96, 128],
}

candidates = []
for c1, c2, c3 in product(*search_space.values()):
    candidate = EdgeCNN(c1=c1, c2=c2, c3=c3)
    params = count_parameters(candidate)

    # Proxy objective until real hardware measurements exist.
    # Replace this with measured latency/energy on the target edge device.
    estimated_cost = params

    candidates.append({
        "c1": c1, "c2": c2, "c3": c3,
        "params": params,
        "size_MB_FP32": parameter_bytes(candidate) / (1024**2),
        "proxy_cost": estimated_cost
    })

nas_df = pd.DataFrame(candidates).sort_values("proxy_cost")
nas_df.head(10)


### The important NAS extension: hardware-aware search

Do **not** assume:

```text
fewer FLOPs = faster
```

on a real edge device.

Latency depends on:

- accelerator architecture
- memory bandwidth
- cache behavior
- operator/kernel implementation
- tensor layouts
- supported quantization formats
- compiler fusion
- CPU/NPU/GPU partition boundaries
- batch size
- thermal/power state

A good edge NAS loop therefore looks like:

```text
candidate architecture
        ↓
train / weight-sharing estimate
        ↓
export graph
        ↓
compile for target
        ↓
run on hardware
        ↓
latency + RAM + power + accuracy
        ↓
update search
```

That turns NAS into **hardware-aware architecture search** rather than a purely theoretical FLOP search.


# 11. Compression sequencing — compare strategies

A useful experimental matrix is:

```text
A: baseline → quantization
B: baseline → pruning → fine-tune → quantization
C: baseline → distillation → quantization
D: baseline → structured pruning → distillation → quantization
E: NAS → distillation → structured pruning → quantization
F: NAS → quantization-aware training
```

The correct sequence depends on the model and hardware.

A production experiment should record:

```text
accuracy
parameter count
model file size
peak RAM
MACs/FLOPs
cold-start time
P50 latency
P95 latency
throughput
energy/inference
thermal behavior
operator fallback count
```


In [ ]:
# 12. Generic benchmark harness.
# Replace dummy inference with your target dataset and hardware.

@torch.inference_mode()
def benchmark(model, input_shape=(1, 3, 32, 32), runs=100, warmup=10):
    model = model.eval()
    x = torch.randn(*input_shape, device=next(model.parameters()).device)

    for _ in range(warmup):
        _ = model(x)

    if x.is_cuda:
        torch.cuda.synchronize()
    start = time.perf_counter()

    for _ in range(runs):
        _ = model(x)

    if x.is_cuda:
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start
    return {
        "avg_ms": elapsed / runs * 1000,
        "throughput_per_sec": runs / elapsed
    }

benchmark(model, runs=30)


# 13. Model partitioning — split the graph across edge accelerators

After compression, the model still has to execute.

A conceptual graph may look like:

```text
Input
  │
  ▼
Preprocess ─────────────── CPU
  │
  ▼
Conv/Attention ────────── NPU
  │
  ▼
Unsupported custom op ─── CPU
  │
  ▼
Postprocess ───────────── GPU/CPU
```

This is **partitioning**.

The compiler identifies supported subgraphs and delegates them to hardware-specific backends.

### Python control plane

Python can define/export/configure the graph.

### Lower-level implementation

The actual kernels/runtime often involve C++, C, CUDA, vendor SDKs, or accelerator-specific code.

So Python is absolutely useful for the workflow, but it cannot magically replace a vendor's hardware backend.


## 13.1 ExecuTorch

**ExecuTorch is the PyTorch edge path I would use as the first concrete Python example.**

The documented flow is essentially:

```text
PyTorch model
     ↓
torch.export
     ↓
edge graph
     ↓
quantization / graph transformations
     ↓
partitioner / backend lowering
     ↓
.pte program
     ↓
ExecuTorch runtime
     ↓
device
```

Its documentation explicitly describes export, compilation, quantization, backend delegation, memory planning and runtime execution. citeturn0search2turn0search4

Example skeleton:

```python
# Illustrative — exact APIs can change with the installed ExecuTorch release.

import torch
from torch.export import export

example = (torch.randn(1, 3, 32, 32),)

exported = export(model.eval(), example)

# Then use the target backend's partitioner/lowering configuration.
# Example pattern:
#
# from executorch.exir import to_edge_transform_and_lower
# from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
#
# edge = to_edge_transform_and_lower(
#     exported,
#     partitioner=[XnnpackPartitioner()]
# )
#
# program = edge.to_executorch()
# with open("edge_model.pte", "wb") as f:
#     f.write(program.buffer)

# NOTE:
# Backend APIs are version-specific; pin the ExecuTorch release in production.
```

ExecuTorch's current examples also show XNNPACK lowering and backend-specific PT2E quantization. citeturn0search7


# 14. Apache TVM — the other major Python-first edge/compiler route

TVM is especially interesting if your goal is **compiler-level control and hardware portability**.

Its current tutorials describe:

```text
PyTorch / ONNX / other model
          ↓
      IRModule
          ↓
graph optimizations
          ↓
tensor-program optimization
          ↓
library dispatch / codegen
          ↓
target compilation
          ↓
CPU / GPU / accelerator
```

TVM emphasizes Python-first customization and composable optimization pipelines. citeturn0search11turn0search9

Current TVM tutorials also demonstrate end-to-end optimization of PyTorch ResNet and LLM deployment workflows. citeturn0search5turn0search3


In [ ]:
# TVM integration point — intentionally guarded because TVM is optional.
# This is where Python hands the model to the compiler.

try:
    import tvm
    print("TVM available:", tvm.__version__)
except ImportError:
    print("TVM not installed.")
    print("Install the appropriate Apache TVM build for your platform.")
    print("The actual target-specific compiler/codegen may require additional native dependencies.")

# Conceptual:
#
# exported_program = torch.export.export(model, example_inputs)
# mod = tvm.relax.frontend.torch.from_exported_program(exported_program)
# mod = tvm.relax.get_pipeline("zero")(mod)
# executable = tvm.compile(mod, target="llvm")  # target changes with hardware


# 15. A practical edge optimization controller

This is the architecture I would use for an MLOps/AI-engineering implementation:

```python
class EdgeOptimizer:
    profile()
    search_architecture()
    distill()
    prune_structured()
    quantize()
    export()
    partition()
    compile()
    benchmark_on_device()
    validate_accuracy()
    select_pareto_frontier()
```

The important idea is that **compression is not one algorithm**. It is an optimization loop with measurements at every stage.


In [ ]:
# 16. Python orchestration skeleton

class EdgeOptimizer:
    def __init__(self, model, device="cpu"):
        self.model = model.to(device)
        self.device = device
        self.history = []

    def record(self, stage, model, metrics=None):
        row = summarize_model(model, stage)
        if metrics:
            row.update(metrics)
        self.history.append(row)

    def structured_prune(self, keep=(0.75, 0.75, 0.75)):
        self.model = prune_edgecnn_channels(self.model, *keep)
        self.record("structured_pruned")
        return self.model

    def quantize(self):
        # For a real CNN use a backend-appropriate PT2E/static/QAT workflow.
        self.model = torch.ao.quantization.quantize_dynamic(
            self.model.cpu(), {nn.Linear}, dtype=torch.qint8
        )
        self.record("quantized")
        return self.model

    def report(self):
        return pd.DataFrame(self.history)

optimizer = EdgeOptimizer(copy.deepcopy(model))
optimizer.record("baseline")
optimizer.structured_prune()
optimizer.report()


# 17. LLM / multimodal production optimization strategy

For very large transformer or multimodal systems, the same principles remain but the objects change.

### Pruning

Instead of only CNN channels:

```text
CNN → filters/channels
Transformer → heads / MLP neurons / blocks / experts
MoE → experts / expert routing capacity
KV cache → tokens / dimensions / precision
```

### Distillation

```text
large teacher
      ↓
smaller dense model
      ↓
task / instruction / reasoning / representation distillation
```

### Quantization

```text
weights       → INT8 / INT4 / FP8 / lower precision
activations   → INT8 / FP8
KV cache      → INT8 / lower precision
embeddings    → lower precision where tolerated
```

### Architecture optimization

```text
dense → MoE
large → small
long context → local/global attention
full KV cache → compressed/sliding-window cache
uniform width → layer-wise width allocation
```

### Edge-specific compilation

```text
attention kernels
        ↓
fused kernels
        ↓
accelerator-specific codegen
        ↓
memory planning
        ↓
operator partitioning
```

For an "AGI" system, think of this as **capacity allocation across the entire inference stack**, not merely shrinking neural-network weights.


# 18. Compression decision matrix

| Technique | Removes capacity? | Changes precision? | Physically shrinks dense tensors? | Can reduce compute? | Accuracy risk | Edge relevance |
|---|---:|---:|---:|---:|---|---|
| Unstructured pruning | Yes | No | Usually no | Backend-dependent | Medium | Medium |
| Structured channel pruning | Yes | No | **Yes** | **Yes** | Medium | **High** |
| Knowledge distillation | Reallocates capacity | No | Through student | Yes | Medium | **High** |
| INT8 PTQ | No | **Yes** | **Yes** | **Yes** on supported HW | Low–Medium | **Very high** |
| INT8 QAT | No | **Yes** | **Yes** | **Yes** | Lower than PTQ in many cases | High |
| INT4 weight-only | No | **Yes** | **Yes** | Backend-dependent | Medium–High | High for LLMs |
| NAS | **Yes / reallocates** | Optional | Yes | Yes | Search-dependent | **Very high** |
| Graph fusion | No | No | Sometimes | **Yes** | Usually low | **Very high** |
| Partitioning | No | No | No | **Yes** | Low | **Very high** |
| Compiler/codegen | No | No | No | **Yes** | Low | **Very high** |

The key lesson:

> **Model compression and model compilation are complementary.**

A 50% smaller model can still be slow if it maps badly to the accelerator. A slightly larger model can sometimes be faster because its operators fuse better or match the hardware's preferred tensor shapes.


# 19. Production experiment checklist

For every candidate model, persist a row like:

```text
model_id
git_sha
dataset_version
teacher_id
student_architecture
pruning_method
pruning_ratio
quantization_scheme
calibration_dataset
compiler_version
runtime_version
target_device
backend
operator_fallbacks
parameter_count
file_size_bytes
peak_ram_bytes
MACs
FLOPs
accuracy
P50_latency_ms
P95_latency_ms
throughput
energy_per_inference
temperature
```

This turns compression into a reproducible **MLOps optimization experiment** rather than a one-off model tweak.

### Recommended next implementation

For your broader Python/MLOps work, the natural progression is:

```text
PyTorch
  ↓
Torch-Pruning / custom structured pruning
  ↓
distillation training
  ↓
TorchAO / PT2E quantization
  ↓
torch.export
  ↓
ExecuTorch or TVM
  ↓
hardware-specific partitioning
  ↓
Prometheus/Grafana benchmark telemetry
  ↓
MLflow experiment + model registry
```

That gives you a genuine **model-compression → compiler → edge-runtime pipeline** rather than just a collection of compression demos.


# 20. ExecuTorch — complete Python SDK path

This section completes the lab with the current Python-driven ExecuTorch workflow:

```text
PyTorch nn.Module
 → structured pruning / distillation / low-rank optimization
 → torch.export
 → backend-aware PT2E quantization
 → ExecuTorch partition + lowering
 → .pte edge program
 → CPU / GPU / NPU backend
```

Python can drive essentially the whole optimization and export control plane. Native/vendor code is still used underneath for optimized kernels, accelerator compilers, runtime and firmware.


In [ ]:
# Optional installation — uncomment in a fresh compatible environment.
# %pip install -U torch torchvision torchao executorch
#
# Restart the kernel after installation if necessary.


In [ ]:
# 21. Verify ExecuTorch Python SDK
def check_executorch():
    try:
        import executorch
        from executorch.exir import to_edge_transform_and_lower
        from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
        print("ExecuTorch Python SDK available")
        return True
    except Exception as exc:
        print("ExecuTorch not available in this kernel:", repr(exc))
        return False

HAS_EXECUTORCH = check_executorch()


## 22. FP32 PyTorch → ExecuTorch `.pte`

`XnnpackPartitioner` finds graph regions that XNNPACK can execute and lowers/delegates them. The resulting `.pte` is an ExecuTorch program rather than a PyTorch checkpoint.


In [ ]:
from pathlib import Path

def export_xnnpack_pte(model, example_inputs, output="edgecnn_xnnpack_fp32.pte"):
    if not HAS_EXECUTORCH:
        print("Skipping: install ExecuTorch first.")
        return None

    from executorch.exir import to_edge_transform_and_lower
    from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

    cpu_model = copy.deepcopy(model).cpu().eval()
    cpu_inputs = tuple(x.cpu() for x in example_inputs)

    exported = torch.export.export(cpu_model, cpu_inputs)
    edge = to_edge_transform_and_lower(
        exported,
        partitioner=[XnnpackPartitioner()],
    )
    program = edge.to_executorch()

    output = Path(output)
    with output.open("wb") as f:
        f.write(program.buffer)

    print("Created:", output.resolve())
    print("PTE bytes:", output.stat().st_size)
    return output

example_inputs = (torch.randn(1, 3, 32, 32),)
fp32_pte = export_xnnpack_pte(
    structured, example_inputs, "edgecnn_structured_xnnpack_fp32.pte"
)


# 23. PT2E INT8 quantization → XNNPACK

This is the more complete edge path:

```text
pruned model
 → torch.export
 → XNNPACKQuantizer
 → prepare_pt2e
 → representative calibration
 → convert_pt2e
 → torch.export
 → XNNPACK partition/lowering
 → quantized .pte
```

The key is **backend-aware quantization**: the quantizer chooses representations the target backend can actually exploit.


In [ ]:
def pt2e_xnnpack_quantize(model, example_inputs, calibration_batches=8):
    if not HAS_EXECUTORCH:
        print("Skipping: install ExecuTorch first.")
        return None

    from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
    from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import (
        XNNPACKQuantizer,
        get_symmetric_quantization_config,
    )

    model = copy.deepcopy(model).cpu().eval()
    inputs = tuple(x.cpu() for x in example_inputs)

    graph_module = torch.export.export(model, inputs).module()

    quantizer = XNNPACKQuantizer()
    quantizer.set_global(
        get_symmetric_quantization_config(is_per_channel=True)
    )

    prepared = prepare_pt2e(graph_module, quantizer)

    # Replace synthetic samples with representative production calibration data.
    with torch.no_grad():
        for _ in range(calibration_batches):
            prepared(torch.randn_like(inputs[0]))

    return convert_pt2e(prepared)

q_model = pt2e_xnnpack_quantize(structured, example_inputs)

if q_model is not None:
    print(q_model.graph)


In [ ]:
def export_quantized_xnnpack_pte(
    quantized_model, example_inputs, output="edgecnn_xnnpack_int8.pte"
):
    if not HAS_EXECUTORCH or quantized_model is None:
        print("Skipping quantized export.")
        return None

    from executorch.exir import to_edge_transform_and_lower
    from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

    inputs = tuple(x.cpu() for x in example_inputs)
    exported = torch.export.export(quantized_model, inputs)

    edge = to_edge_transform_and_lower(
        exported,
        partitioner=[XnnpackPartitioner()],
    )
    program = edge.to_executorch()

    output = Path(output)
    with output.open("wb") as f:
        f.write(program.buffer)

    print("Created:", output.resolve())
    print("PTE bytes:", output.stat().st_size)
    return output

int8_pte = export_quantized_xnnpack_pte(
    q_model, example_inputs, "edgecnn_structured_int8_xnnpack.pte"
)


# 24. Accuracy gate

Compression is an optimization **subject to an accuracy constraint**, not simply a contest to produce the smallest file.

For classification, a typical acceptance rule is:

```text
compressed_accuracy ≥ baseline_accuracy − allowed_accuracy_drop
```

For an LLM, add perplexity/task scores, retrieval quality, grounding, reasoning and safety evaluations.


In [ ]:
@torch.inference_mode()
def classification_accuracy(model, loader, device="cpu"):
    model = model.to(device).eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.numel()
    return correct / max(total, 1)

# baseline_acc = classification_accuracy(model, test_loader, device)
# compressed_acc = classification_accuracy(structured, test_loader, device)


# 25. Reusable distillation training

A strong practical sequence is:

```text
teacher
 → choose/prune student architecture
 → distill
 → fine-tune
 → quantize
 → compile
```

Distillation after structural compression gives the smaller network a chance to adapt to the capacity that was actually removed.


In [ ]:
def train_student_with_distillation(
    teacher, student, train_loader, epochs=3, lr=1e-3,
    temperature=4.0, alpha=0.5, device=device
):
    teacher = teacher.to(device).eval()
    student = student.to(device).train()
    opt = torch.optim.AdamW(student.parameters(), lr=lr)
    history = []

    for epoch in range(epochs):
        total_loss = examples = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            with torch.no_grad():
                teacher_logits = teacher(x)

            student_logits = student(x)
            loss = distillation_loss(
                student_logits, teacher_logits, y,
                T=temperature, alpha=alpha
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item() * x.size(0)
            examples += x.size(0)

        epoch_loss = total_loss / max(examples, 1)
        history.append(epoch_loss)
        print(f"epoch={epoch+1} distillation_loss={epoch_loss:.4f}")

    return student, history


# 26. What does neuron pruning physically remove?

For a two-layer MLP:

```text
Linear(d_model → d_hidden)
        ↓
 hidden neurons
        ↓
Linear(d_hidden → d_model)
```

Removing hidden neuron `j` requires removing **row j** from the first weight matrix and **column j** from the second. That physically reduces the hidden width rather than merely masking a value.


In [ ]:
class TinyMLP(nn.Module):
    def __init__(self, d_in=16, d_hidden=64, d_out=8):
        super().__init__()
        self.up = nn.Linear(d_in, d_hidden)
        self.down = nn.Linear(d_hidden, d_out)

    def forward(self, x):
        return self.down(F.gelu(self.up(x)))

def prune_mlp_hidden_neurons(src, keep_ratio=0.75):
    src = copy.deepcopy(src).cpu().eval()
    incoming = src.up.weight.detach().abs().sum(1)
    outgoing = src.down.weight.detach().abs().sum(0)
    score = incoming + outgoing

    k = max(1, int(src.up.out_features * keep_ratio))
    keep = torch.topk(score, k).indices.sort().values
    dst = TinyMLP(src.up.in_features, k, src.down.out_features)

    with torch.no_grad():
        dst.up.weight.copy_(src.up.weight[keep])
        dst.up.bias.copy_(src.up.bias[keep])
        dst.down.weight.copy_(src.down.weight[:, keep])
        dst.down.bias.copy_(src.down.bias)

    return dst, keep

mlp = TinyMLP()
small_mlp, kept = prune_mlp_hidden_neurons(mlp, 0.50)
print("Parameters:", count_parameters(mlp), "→", count_parameters(small_mlp))
print("Hidden neurons:", mlp.up.out_features, "→", small_mlp.up.out_features)


# 27. Transformer attention-head scoring

A transformer head is a structured slice of Q/K/V. Production head removal is architecture-specific because models may use separate Q/K/V, fused QKV, GQA/MQA or tensor-parallel sharding.

The generic Python step is therefore to **score heads**, then perform a model-specific structural rewrite.


In [ ]:
def attention_head_l1_scores(qkv_weight, num_heads):
    # Demonstration for fused QKV with shape [3*d_model, d_model].
    three_d, _ = qkv_weight.shape
    assert three_d % 3 == 0
    d_model = three_d // 3
    assert d_model % num_heads == 0

    head_dim = d_model // num_heads
    q, k, v = qkv_weight.chunk(3, dim=0)
    scores = []

    for h in range(num_heads):
        sl = slice(h * head_dim, (h + 1) * head_dim)
        scores.append(q[sl].abs().sum() + k[sl].abs().sum() + v[sl].abs().sum())

    return torch.stack(scores)

fake_qkv = torch.randn(3 * 64, 64)
head_scores = attention_head_l1_scores(fake_qkv, num_heads=8)
print("Head scores:", head_scores)
print("Lowest importance head:", int(torch.argmin(head_scores)))


## Structured pruning inside a transformer

An LLM does not normally have CNN-style image channels, but the same principle appears in transformer structures.

A multi-head attention projection commonly has hidden width:

```text
hidden_size = num_heads × head_dim
```

Example:

```text
hidden_size = 768
num_heads   = 12
head_dim    = 64
```

Removing one attention head conceptually removes a 64-dimensional slice from the head representation. A production implementation must also keep the surrounding projection dimensions and graph consistent.

MLP structured pruning works similarly: remove entire intermediate neurons instead of individual scalar weights.


In [ ]:
# Tiny transformer-style attention-head anatomy in Python.

import torch

batch = 2
seq = 8
hidden = 768
num_heads = 12
head_dim = hidden // num_heads

x = torch.randn(batch, seq, hidden)
Wq = torch.randn(hidden, hidden)

Q = x @ Wq
Q_heads = Q.view(batch, seq, num_heads, head_dim)

print("Q:", Q.shape)
print("Q split into heads:", Q_heads.shape)

# Remove head 5 from the explicit head representation.
keep_heads = [h for h in range(num_heads) if h != 5]
Q_pruned_heads = Q_heads[:, :, keep_heads, :]

print("after removing one head:", Q_pruned_heads.shape)
print("remaining head dimensions:", len(keep_heads) * head_dim)

# Important:
# This demonstrates the structural dimension removal.
# A deployable transformer also needs compatible Q/K/V and output-projection
# weights rebuilt/fine-tuned around the new architecture.


In [ ]:
# Structured MLP-neuron pruning:
# physically select surviving intermediate neurons.

import torch
import torch.nn as nn

hidden = 256
intermediate = 1024

up = nn.Linear(hidden, intermediate)
down = nn.Linear(intermediate, hidden)

# Score each intermediate neuron by the magnitude of its incoming weights.
scores = up.weight.detach().abs().sum(dim=1)

keep_count = 768
keep = torch.topk(scores, keep_count).indices.sort().values

new_up = nn.Linear(hidden, keep_count)
new_down = nn.Linear(keep_count, hidden)

with torch.no_grad():
    # Keep selected output neurons of the up projection.
    new_up.weight.copy_(up.weight[keep])
    new_up.bias.copy_(up.bias[keep])

    # Remove corresponding input columns from the down projection.
    new_down.weight.copy_(down.weight[:, keep])
    new_down.bias.copy_(down.bias)

print("old up:", up.weight.shape)
print("new up:", new_up.weight.shape)
print("old down:", down.weight.shape)
print("new down:", new_down.weight.shape)

z = torch.randn(4, hidden)
out = new_down(torch.relu(new_up(z)))
print("output:", out.shape)


# 28. Low-rank approximation

For a dense matrix `W[out, in]`, truncated SVD approximates it with two smaller matrices of rank `r`:

```text
W ≈ Uᵣ Σᵣ Vᵣᵀ

original parameters   = out × in
factorized parameters ≈ out × r + r × in
```

This is a genuine compression when `r` is sufficiently small.


In [ ]:
class LowRankLinear(nn.Module):
    def __init__(self, first, second, bias=None):
        super().__init__()
        self.first = first
        self.second = second
        if bias is None:
            self.register_parameter("bias", None)
        else:
            self.bias = nn.Parameter(bias.clone())

    def forward(self, x):
        y = self.second(self.first(x))
        return y if self.bias is None else y + self.bias

def svd_factorize_linear(layer, rank):
    W = layer.weight.detach().cpu()
    U, S, Vh = torch.linalg.svd(W, full_matrices=False)
    rank = min(rank, len(S))

    first = nn.Linear(layer.in_features, rank, bias=False)
    second = nn.Linear(rank, layer.out_features, bias=False)

    with torch.no_grad():
        first.weight.copy_(Vh[:rank])
        second.weight.copy_(U[:, :rank] * S[:rank].unsqueeze(0))

    return LowRankLinear(first, second, layer.bias)

dense = nn.Linear(512, 512)
lowrank = svd_factorize_linear(dense, rank=64)
print("Dense:", count_parameters(dense))
print("Low rank:", count_parameters(lowrank))
print("Compression ratio:", count_parameters(dense) / count_parameters(lowrank))


# 29. Hardware-aware NAS

Do not optimize NAS solely against FLOPs. Measure the candidate on the actual target whenever possible because operator implementations, memory traffic, tensor shape and accelerator utilization can dominate.


In [ ]:
def architecture_candidates():
    for c1 in [16, 24, 32]:
        for c2 in [32, 48, 64]:
            for c3 in [64, 96, 128]:
                yield {"c1": c1, "c2": c2, "c3": c3}

def hardware_aware_search(max_candidates=10):
    rows = []
    for i, cfg in enumerate(architecture_candidates()):
        if i >= max_candidates:
            break
        candidate = EdgeCNN(**cfg).cpu().eval()
        perf = benchmark(candidate, runs=20, warmup=5)
        rows.append({
            **cfg,
            "parameters": count_parameters(candidate),
            "latency_ms": perf["avg_ms"],
            "throughput": perf["throughput_per_sec"],
            "accuracy": np.nan,  # populate after candidate training/evaluation
        })
    return pd.DataFrame(rows).sort_values("latency_ms")

nas_latency_results = hardware_aware_search()
nas_latency_results


# 30. Pareto selection

The best edge model is rarely just the smallest. A candidate belongs on the Pareto frontier when no other candidate is simultaneously more accurate, faster and smaller.


In [ ]:
def pareto_frontier(df, accuracy_col="accuracy",
                    latency_col="latency_ms", size_col="parameters"):
    clean = df.dropna(subset=[accuracy_col, latency_col, size_col]).copy()
    keep = []

    for i, row in clean.iterrows():
        dominated = False
        for j, other in clean.iterrows():
            if i == j:
                continue
            no_worse = (
                other[accuracy_col] >= row[accuracy_col]
                and other[latency_col] <= row[latency_col]
                and other[size_col] <= row[size_col]
            )
            strictly_better = (
                other[accuracy_col] > row[accuracy_col]
                or other[latency_col] < row[latency_col]
                or other[size_col] < row[size_col]
            )
            if no_worse and strictly_better:
                dominated = True
                break
        if not dominated:
            keep.append(i)

    return clean.loc[keep]


# 31. Partitioning and the Python/native boundary

Python can perform capture, quantization configuration, graph transformation, partition selection, lowering and serialization.

The backend then owns optimized execution:

```text
XNNPACK   → mobile/edge CPU
Core ML   → Apple hardware
QNN       → Qualcomm accelerators
Vulkan    → supported GPU path
other NPU → backend/vendor-specific
```

A call such as:

```python
to_edge_transform_and_lower(
    exported,
    partitioner=[XnnpackPartitioner()]
)
```

is therefore a Python-controlled **graph partition + lowering decision**, even though the resulting kernels are native.


# 32. Qualcomm/NPU boundary example

ExecuTorch also has a QNN backend and backend-specific quantization for Qualcomm hardware. Python can configure the graph and quantization, but a supported Qualcomm SDK/toolchain and target device are still required.

Conceptually:

```python
# Version/target-specific pseudocode:
quantizer = QnnQuantizer(...)
prepared = prepare_pt2e(exported.module(), quantizer)
calibrate(prepared)
quantized = convert_pt2e(prepared)

program = to_edge_transform_and_lower(
    torch.export.export(quantized, inputs),
    partitioner=[QnnPartitioner(...)]
).to_executorch()
```

This notebook intentionally leaves target-specific constructor arguments as comments because chipset and QNN/ExecuTorch versions determine them.


# 33. LLM production compression — edge is one deployment target

The same pipeline scales conceptually to transformers:

```text
large teacher
 → smaller architecture / NAS
 → knowledge distillation
 → MLP-neuron pruning
 → attention-head / block pruning
 → low-rank factorization
 → fine-tuning
 → INT8 / INT4 / mixed precision
 → KV-cache quantization/compression
 → torch.export
 → ExecuTorch
 → partition/lower
 → CPU/GPU/NPU
```

For LLMs, memory bandwidth and KV-cache growth can matter as much as raw arithmetic.


In [ ]:
# 34. Approximate transformer weight + KV-cache memory
def transformer_memory_estimate(
    parameter_count, weight_bits, layers, kv_heads,
    head_dim, sequence_length, batch_size=1, kv_bits=16
):
    weight_bytes = parameter_count * weight_bits / 8
    kv_elements = (
        2 * layers * batch_size * kv_heads
        * sequence_length * head_dim
    )
    kv_bytes = kv_elements * kv_bits / 8
    return {
        "weights_GB": weight_bytes / 1e9,
        "kv_cache_GB": kv_bytes / 1e9,
        "total_GB_approx": (weight_bytes + kv_bytes) / 1e9,
    }

memory_table = pd.DataFrame([
    {"configuration": "FP16 weights + FP16 KV",
     **transformer_memory_estimate(7_000_000_000, 16, 32, 8, 128, 4096, kv_bits=16)},
    {"configuration": "INT4 weights + FP16 KV",
     **transformer_memory_estimate(7_000_000_000, 4, 32, 8, 128, 4096, kv_bits=16)},
    {"configuration": "INT4 weights + INT8 KV",
     **transformer_memory_estimate(7_000_000_000, 4, 32, 8, 128, 4096, kv_bits=8)},
])
memory_table


# 35. Complete automated optimization controller

Now combine the pieces:

```text
PROFILE
 → ARCHITECTURE SEARCH
 → STRUCTURED PRUNING
 → DISTILL/FINE-TUNE
 → QUANTIZE
 → EXPORT
 → PARTITION/LOWER
 → DEVICE BENCHMARK
 → ACCURACY GATE
 → PARETO SELECTION
```


In [ ]:
from dataclasses import dataclass, asdict

@dataclass
class EdgeConstraints:
    max_model_mb: float = 50.0
    max_latency_ms: float = 20.0
    max_accuracy_drop: float = 0.01

@dataclass
class CompressionConfig:
    channel_keep: tuple = (0.75, 0.75, 0.75)
    distill_temperature: float = 4.0
    distill_alpha: float = 0.5
    quantization: str = "int8"
    backend: str = "xnnpack"

class CompleteEdgePipeline:
    def __init__(self, teacher, config, constraints):
        self.teacher = copy.deepcopy(teacher)
        self.config = config
        self.constraints = constraints
        self.student = None
        self.artifacts = {}

    def prune(self):
        self.student = prune_edgecnn_channels(
            self.teacher, *self.config.channel_keep
        )
        return self.student

    def distill(self, train_loader, epochs=3):
        if self.student is None:
            self.prune()
        self.student, history = train_student_with_distillation(
            self.teacher, self.student, train_loader,
            epochs=epochs,
            temperature=self.config.distill_temperature,
            alpha=self.config.distill_alpha,
        )
        self.artifacts["distillation_history"] = history
        return self.student

    def quantize_pt2e(self, example_inputs):
        if self.student is None:
            self.prune()
        q = pt2e_xnnpack_quantize(self.student.cpu(), example_inputs)
        self.artifacts["quantized_model"] = q
        return q

    def export_pte(self, example_inputs, output="optimized_model.pte"):
        q = self.artifacts.get("quantized_model")
        if q is not None:
            path = export_quantized_xnnpack_pte(q, example_inputs, output)
        else:
            path = export_xnnpack_pte(self.student, example_inputs, output)
        self.artifacts["pte"] = path
        return path

    def summary(self):
        return {
            "config": asdict(self.config),
            "constraints": asdict(self.constraints),
            "student_parameters": (
                count_parameters(self.student)
                if self.student is not None else None
            ),
        }

pipeline = CompleteEdgePipeline(
    model.cpu(), CompressionConfig(), EdgeConstraints()
)
pipeline.prune()
pipeline.summary()

# Real workflow:
# pipeline.distill(train_loader, epochs=10)
# pipeline.quantize_pt2e(example_inputs)
# pipeline.export_pte(example_inputs, "optimized_edgecnn.pte")


# 36. Python capability map

| Stage | Python? | Typical mechanism |
|---|---|---|
| Model definition | ✅ | PyTorch |
| Weight/channel/neuron pruning | ✅ | PyTorch/custom libraries |
| Distillation | ✅ | PyTorch training |
| Low-rank approximation | ✅ | `torch.linalg` |
| NAS controller | ✅ | Python/search framework |
| PTQ/QAT configuration | ✅ | TorchAO/PT2E |
| Graph capture | ✅ | `torch.export` |
| ExecuTorch lowering | ✅ | Python API |
| Backend partition selection | ✅ | Python partitioner |
| `.pte` generation | ✅ | Python API |
| Benchmark orchestration | ✅ | Python + device bridge |
| Optimized runtime kernels | native underneath | C/C++/Metal/etc. |
| Vendor NPU compiler | vendor-specific | SDK/toolchain |
| Custom accelerator kernels | often native | C++/CUDA/Metal/vendor APIs |

**Therefore:** Python can be the control plane for the complete optimization pipeline. It does not need to replace the low-level runtime or accelerator implementation.


# 37. Final production optimization architecture

```text
                 PYTHON / MLOps CONTROL PLANE

 PyTorch teacher
       │
       ├── NAS / hardware-aware search ◄──── device metrics
       ▼
 structured pruning
 channels / neurons / heads / blocks
       │
       ▼
 knowledge distillation
       │
       ▼
 low-rank approximation
       │
       ▼
 PT2E / TorchAO quantization
       │
       ▼
 torch.export
       │
       ▼
 ExecuTorch
 graph transforms
 partitioning
 backend lowering
       │
       ▼
      .pte
       │
 ┌─────┼─────────────┐
 ▼     ▼             ▼
CPU   GPU          NPU/DSP
 │     │             │
 └─────┴──── metrics ┘
       │
       ▼
 accuracy + latency + RAM + energy + thermal
       │
       └──────────────► architecture/NAS feedback loop
```

The important engineering conclusion is that **edge optimization is a system problem**:

```text
architecture
× structured sparsity
× numerical precision
× memory movement
× compiler transformations
× accelerator operator support
× runtime behavior
```

The best compressed model is the model that meets the accuracy target on the **actual deployment hardware**, not merely the model with the fewest parameters.


# 38. Production LLM optimization: what you optimize in practice

For production LLM serving, optimization is broader than model-file compression.

You care about:

```text
weights
activations
attention compute
KV cache
memory bandwidth
batching
prefill latency
decode latency
tokens / second
GPU utilization
kernel fusion
tensor parallelism
pipeline parallelism
quantization format
runtime support
```

A useful distinction:

```text
MODEL optimization
    pruning / distillation / low rank / quantization

RUNTIME optimization
    batching / cache reuse / fused kernels / graph compilation

SYSTEM optimization
    replicas / autoscaling / scheduling / GPU placement / observability
```


In [ ]:
# A tiny production-style memory calculator for model weights.

def weight_memory_gb(parameter_count, bits_per_weight):
    return parameter_count * bits_per_weight / 8 / (1024**3)

params = 7_000_000_000  # approximate 7B parameter model

for bits in [32, 16, 8, 4]:
    print(f"{bits:>2}-bit weights: {weight_memory_gb(params, bits):.2f} GiB")

# This is weight storage only.
# Runtime memory also includes activations, KV cache, allocator overhead,
# temporary buffers, framework/runtime overhead, etc.


## KV cache — optimization beyond the weights

During autoregressive inference, the model repeatedly needs keys and values from previous tokens. Instead of recomputing them, production runtimes maintain a **KV cache**.

Its size grows with context length and concurrent sequences, so a model that "fits" by weight size can still run out of GPU memory under real traffic.


In [ ]:
# Simplified KV-cache size estimator.

def kv_cache_gib(
    layers,
    batch,
    sequence_length,
    kv_heads,
    head_dim,
    bytes_per_element=2,  # FP16/BF16
):
    # K and V => factor of 2
    elements = (
        layers
        * batch
        * sequence_length
        * kv_heads
        * head_dim
        * 2
    )
    return elements * bytes_per_element / (1024**3)

print(
    "KV cache GiB:",
    kv_cache_gib(
        layers=32,
        batch=8,
        sequence_length=8192,
        kv_heads=8,
        head_dim=128,
        bytes_per_element=2,
    )
)


## Optimization must end in measurement

An optimization is not successful merely because:

- the parameter count decreased,
- a tensor became sparse,
- the model file became smaller,
- or a theoretical FLOP count dropped.

For a production LLM you should compare at least:

```text
quality / task accuracy
model memory
peak GPU memory
KV-cache memory
time to first token (TTFT)
inter-token latency
tokens/sec
requests/sec
P50/P95/P99 latency
batch-size sensitivity
cost per million tokens
GPU utilization
power/thermal limits where applicable
```


In [ ]:
# Minimal experiment record that can later be written to MLflow/Pandas/etc.

from dataclasses import dataclass, asdict

@dataclass
class OptimizationResult:
    name: str
    precision: str
    parameters_millions: float
    model_memory_gb: float
    ttft_ms: float
    tokens_per_second: float
    quality_score: float

baseline = OptimizationResult(
    name="baseline",
    precision="FP16",
    parameters_millions=7000,
    model_memory_gb=14.0,
    ttft_ms=120.0,
    tokens_per_second=45.0,
    quality_score=0.84,
)

candidate = OptimizationResult(
    name="INT4_candidate",
    precision="INT4",
    parameters_millions=7000,
    model_memory_gb=3.5,
    ttft_ms=92.0,
    tokens_per_second=68.0,
    quality_score=0.825,
)

print(asdict(baseline))
print(asdict(candidate))
